# Robot Performance Measurement and Failure Analysis

### By Ben Toaz

## Data Collection and Preparation

In [ ]:
import pandas as pd
import glob
import os

# Path where the feather files are saved
data_path = "data/aursad"

# Get all feather files, sorted in order (important for time series)
feather_files = sorted(glob.glob(os.path.join(data_path, "part_*.feather")))

# Load and concatenate
num_files = len(feather_files)//10
df_aursad = pd.concat([pd.read_feather(f) for f in feather_files[:num_files]], ignore_index=True)

# super high resolution, need to take limited samples
df_sampled = df_aursad.iloc[::100]

print(f"Loaded {num_files} files.")
print(f"Combined DataFrame shape: {df_aursad.shape}")


Loaded 7 files.
Combined DataFrame shape: (583247, 134)


In [ ]:
df_aursad.head()


In [ ]:
df_aursad.info()

In [ ]:
# Missingness

import matplotlib.pyplot as plt
import seaborn as sns

# plt.figure(figsize=(16,8))
# sns.heatmap(df_aursad.isna(), cmap="magma")

In [ ]:
df_aursad = df_aursad.rename(columns={'timestamp': 'time'})
df_aursad['time'] =  df_aursad['time'] - df_aursad['time'].iloc[0]
df_aursad

In [ ]:
# Renaming to match CobotOps
for i in range(6):
    df_aursad = df_aursad.rename(columns={f'actual_current_{i}': f'Current_J{i}'})
    df_aursad = df_aursad.rename(columns={f'actual_TCP_speed_{i}': f'Speed_J{i}'})
    df_aursad = df_aursad.rename(columns={f'joint_temperatures_{i}': f'Temperature_J{i}'})

# Encode labels for screwing failures
df_aursad = pd.get_dummies(df_aursad, columns=['label'], prefix='label')
label_names = ["Normal operation", "Damaged screw", "Extra assembly component", "Missing screw", "Damaged thread samples", "Screw Loosening"]

for i, label in enumerate(label_names):
    df_aursad = df_aursad.rename(columns={f'label_{i}': label})
df_aursad.head()


In [ ]:
from library import *

fig = histogram_plots(df_aursad)
fig.show()

In [ ]:

def joint_correlation_heatmaps(df, is_cobotops=True):
    feature_type_lst = ["Current", "Speed", "Temperature"]

    # Create 2x3 subplots
    fig = make_subplots(
        rows=3, cols=2,
        subplot_titles=[f'Joint {i}' for i in range(6)],
        vertical_spacing=0.125,
        horizontal_spacing=0.3
    )

    # Loop through joints 0-5
    for joint_idx in range(6):
        # Select columns for this joint
        cols = []
        for feature_type in feature_type_lst:
            cols.append(f"{feature_type}_J{joint_idx}")

        if is_cobotops:
            cols += ['Robot_ProtectiveStop', 'grip_lost', 'cycle', 'Tool_current']
        else:
            # cols +=  ["Normal operation", "Damaged screw", "Extra assembly component", "Missing screw", "Damaged thread samples", "Screw Loosening"]
            cols +=  ["Normal operation", "Damaged screw", "Extra assembly component", "Missing screw", "Screw Loosening"]

        # Calculate correlation
        df_corr = df[cols].corr().round(2)
        
        # Mask upper triangle
        mask = np.zeros_like(df_corr, dtype=bool)
        mask[np.triu_indices_from(mask)] = True
        
        # Apply mask and drop empty rows/cols
        df_corr_viz = df_corr.mask(mask).dropna(how='all').dropna(axis='columns', how='all')

        # Create text array with blanks instead of nan
        text_values = df_corr_viz.values.astype(str)
        text_values[text_values == 'nan'] = ''
        
        # Calculate position in grid
        col = (joint_idx // 3) + 1  # 1 or 2
        row = (joint_idx % 3) + 1   # 1, 2, or 3
        
        # Add heatmap to subplot
        fig.add_trace(
            go.Heatmap(
                z=df_corr_viz.values,
                x=df_corr_viz.columns,
                y=df_corr_viz.index,
                colorscale='Viridis',
                zmid=0,
                text=text_values,
                texttemplate='%{text}',
                textfont={"size": 8},
                showscale=False  # Only show colorbar on last plot
            ),
            row=row, col=col
        )
        
        # Update axes for this subplot
        fig.update_xaxes(tickangle=-45, row=row, col=col)

    fig.update_layout(
        height=1800,
        width=1400,
        showlegend=False,
        margin=dict(l=175, b=150) 
    )
    return fig

fig = joint_correlation_heatmaps(df_aursad, is_cobotops=False)
fig.show()

In [ ]:
def feature_correlation_heatmaps(df_cobots):
    # Correlations by feature type
    feature_type_lst = ["Current", "Speed", "Temperature"]

    feature_pairs = list(combinations(feature_type_lst, 2))

    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=[f'{pair[0]} vs {pair[1]}' for pair in feature_pairs],
        horizontal_spacing=0.125
    )

    for pair_idx, (feat1, feat2) in enumerate(feature_pairs):
        cols = []
        for joint_idx in range(6):
            cols.append(f"{feat1}_J{joint_idx}")
        for joint_idx in range(6):
            cols.append(f"{feat2}_J{joint_idx}")
        
        df_corr = df_cobots[cols].corr().round(2)
        
        # Mask upper triangle
        mask = np.zeros_like(df_corr, dtype=bool)
        mask[np.triu_indices_from(mask)] = True
        
        # Apply mask and drop empty rows/cols
        df_corr_viz = df_corr.mask(mask).dropna(how='all').dropna(axis='columns', how='all')

        # Create text array with blanks instead of nan
        text_values = df_corr_viz.values.astype(str)
        text_values[text_values == 'nan'] = ''
        
        col = pair_idx + 1  # 1, 2, or 3
        
        fig.add_trace(
            go.Heatmap(
                z=df_corr_viz.values,
                x=df_corr_viz.columns,
                y=df_corr_viz.index,
                colorscale='Viridis',
                zmid=0,
                text=text_values,
                texttemplate='%{text}',
                textfont={"size": 8},
                showscale=(pair_idx == 2)
            ),
            row=1, col=col
        )
        
        fig.update_xaxes(tickangle=-45, row=1, col=col)

    fig.update_layout(
        height=450,
    #  title_text="Cross-Feature Correlation Analysis",
        margin=dict(b=100),
        showlegend=False
    )

    return fig

fig = feature_correlation_heatmaps(df_aursad)
fig.show()

In [ ]:
def time_series_plots(df_cobots, error, feature_type):
    # Define column groups
    feature_type_lst = ["Current", "Speed", "Temperature"]
    unit_lst = ["A", "m/s", "Degrees C"]
    colors = px.colors.qualitative.Dark24

    unit = unit_lst[feature_type_lst.index(feature_type)]

    fig_lst = []

    # for feature_type, unit in zip(feature_type_lst, unit):
    cols1 = [f"{feature_type}_J{i}" for i in range(0, 3)]
    cols2 = [f"{feature_type}_J{i}" for i in range(3, 6)]

    # Create subplots
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                        subplot_titles=(f'{feature_type} Joints 0-2', f'{feature_type} Joints 3-5'),
                        vertical_spacing=0.1)

    # Add current traces to first subplot
    for i, col in enumerate(cols1):
        fig.add_trace(go.Scatter(x=df_cobots['time'], y=df_cobots[col],
                                name=col, mode='lines', line=dict(color=colors[i])), row=1, col=1)

    # Add speed traces to second subplot
    for i, col in enumerate(cols2):
        # Avoid black color for last joint
        if i != len(cols2) - 1:
            color = colors[i + 3] 
        else:
            color = colors[6]
        fig.add_trace(go.Scatter(x=df_cobots['time'], y=df_cobots[col], 
                                name=col, mode='lines', line=dict(color=color)), row=2, col=1)

    # Add yellow dots where grip_lost is True
    if error in df_cobots.columns:
        df_flag = df_cobots[df_cobots[error] == True]
        
        if not df_flag.empty:
            # Add markers to subplot 1 (for each joint in cols1)
            for i, col in enumerate(cols1):
                fig.add_trace(go.Scatter(
                    x=df_flag['time'], 
                    y=df_flag[col],
                    mode='markers',
                    marker=dict(color='yellow', size=6, symbol='circle'),
                    name=error,
                    showlegend=(i == 0)  # Only show legend for first occurrence
                ), row=1, col=1)
            
            # Add markers to subplot 2 (for each joint in cols2)
            for i, col in enumerate(cols2):
                fig.add_trace(go.Scatter(
                    x=df_flag['time'], 
                    y=df_flag[col],
                    mode='markers',
                    marker=dict(color='yellow', size=6, symbol='circle'),
                    name=error,
                    showlegend=False  
                ), row=2, col=1)

        # Add rangeslider to bottom subplot only
        fig.update_xaxes(rangeslider_visible=True, row=2, col=1)

        fig.update_xaxes(title_text="Time (s)", row=2, col=1, rangeslider_visible=True)
        fig.update_yaxes(title_text=f"{feature_type} ({unit})", row=1, col=1)
        fig.update_yaxes(title_text=f"{feature_type} ({unit})", row=2, col=1)

        # Update layout
        fig.update_layout(height=800)
        fig_lst.append(fig)

    return fig_lst

error_lst = ["Damaged screw", "Extra assembly component", "Missing screw", "Damaged thread samples"]

fig_lst = time_series_plots(df_aursad, "Damaged screw", "Current")
for fig in fig_lst:
    fig.show()

In [3]:
import pandas as pd

df = pd.read_feather('../data/aursad/aursad_training.feather')

for i in range(6):
    df.rename(columns={f'Speed{i}': f'actual_TCP_speed_{i}'})
    df.rename(columns={f'actual_qd_{i}' : f'Speed{i}'})

df.head()
df.to_feather('../data/aursad/aursad.feather')
